# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.74719097 -0.20593459 -0.79315149  0.10545734  0.61148269]
 [ 0.72838567  0.36621959 -0.32362969 -0.39322666 -0.596077  ]
 [ 0.59104087 -0.13260899  0.74572465 -0.59220079  0.77149838]
 [-0.70713706 -0.7050044   0.73850255 -0.7033699   0.73000025]
 [ 0.41466496  0.99792934 -0.51664477  0.03767511 -0.74483653]
 [ 0.5892851  -0.89822204 -0.39235068 -0.37669106 -0.61629239]
 [ 0.7299874  -0.77367646 -0.24636884 -0.68291155 -0.10731525]
 [ 0.49907277  0.72977437  0.66324554  0.40371991  0.82724503]
 [-0.34032082  0.2989315   0.54625664  0.24661924  0.55417621]
 [ 0.71719284 -0.30738503 -0.21781641 -0.16817508  0.67688839]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a1', 'a2', 'a1', 'a2', 'a1', 'a1', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 1, 0, 1, 0, 0, 0, 0, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.06s/it]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.06s/it, loss=6.2731]

SVI:   6%|▌         | 2/34 [00:01<00:34,  1.06s/it, loss=3.9153]

SVI:   9%|▉         | 3/34 [00:01<00:32,  1.06s/it, loss=2.3588]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.11it/s, loss=2.0943] 

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.11it/s, loss=2.0954]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.11it/s, loss=0.6945]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.76it/s, loss=0.6945]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.76it/s, loss=2.7019]